In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import re
import string

from glob import glob
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
train_path = glob('/kaggle/input/**/train.csv', recursive=True)[0]

train = pd.read_csv(train_path)

print(train.shape)
train.head()

(2000, 8)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [6]:
train.isnull().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

In [7]:
text_columns = ['prompt', 'A', 'B', 'C', 'D', 'E']

train[text_columns] = train[text_columns].fillna('')

In [8]:
def clean_text(text):
    text = text.lower()

    for mark in string.punctuation:
        text = text.replace(mark, ' ')

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [9]:
for column in text_columns:
    train[column + '_clean'] = train[column].apply(clean_text)

train[['prompt', 'prompt_clean']].head()

,prompt,prompt_clean
0,Pick the best possible answer: What is Martin ...,pick the best possible answer what is martin h...
1,What is accelerator-based light-ion fusion?,what is accelerator based light ion fusion
2,Determine the correct option: What is the term...,determine the correct option what is the term ...
3,Select the most accurate option: What is Marti...,select the most accurate option what is martin...
4,Identify the correct statement: What is the co...,identify the correct statement what is the con...


In [10]:
train['prompt_tokens'] = train['prompt_clean'].str.split()

train[['prompt_clean', 'prompt_tokens']].head()

,prompt_clean,prompt_tokens
0,pick the best possible answer what is martin h...,"[pick, the, best, possible, answer, what, is, ..."
1,what is accelerator based light ion fusion,"[what, is, accelerator, based, light, ion, fus..."
2,determine the correct option what is the term ...,"[determine, the, correct, option, what, is, th..."
3,select the most accurate option what is martin...,"[select, the, most, accurate, option, what, is..."
4,identify the correct statement what is the con...,"[identify, the, correct, statement, what, is, ..."


In [11]:
print('Cleaned prompt:')
print(train.loc[0, 'prompt_clean'])

print('\nTokens:')
print(train.loc[0, 'prompt_tokens'])

Cleaned prompt:
pick the best possible answer what is martin heidegger s view on the relationship between time and human existence among the listed options

Tokens:
['pick', 'the', 'best', 'possible', 'answer', 'what', 'is', 'martin', 'heidegger', 's', 'view', 'on', 'the', 'relationship', 'between', 'time', 'and', 'human', 'existence', 'among', 'the', 'listed', 'options']


In [12]:
def average_precision_at_3(actual, predicted):
    predicted = predicted[:3]

    if actual not in predicted:
        return 0

    rank = predicted.index(actual) + 1

    return 1 / rank

In [13]:
def map_at_3(actual_answers, predictions):
    scores = []

    for actual, predicted in zip(actual_answers, predictions):
        score = average_precision_at_3(actual, predicted)
        scores.append(score)

    return np.mean(scores)

In [14]:
actual = 'B'
predicted = ['D', 'B', 'E']

print('AP@3:', average_precision_at_3(actual, predicted))

AP@3: 0.5


In [15]:
clean_columns = [
    'prompt_clean',
    'A_clean',
    'B_clean',
    'C_clean',
    'D_clean',
    'E_clean'
]

all_text = []

for column in clean_columns:
    all_text.extend(train[column].tolist())

print('Number of text samples:', len(all_text))

Number of text samples: 12000


In [16]:
tfidf = TfidfVectorizer(stop_words='english')

tfidf.fit(all_text)

print('TF-IDF vocabulary size:', len(tfidf.vocabulary_))

TF-IDF vocabulary size: 2762


In [17]:
option_labels = ['A', 'B', 'C', 'D', 'E']
tfidf_predictions = []
tfidf_scores = []

for _, row in train.iterrows():
    prompt_vector = tfidf.transform([row['prompt_clean']])

    options = [
        row['A_clean'],
        row['B_clean'],
        row['C_clean'],
        row['D_clean'],
        row['E_clean']
    ]

    option_vectors = tfidf.transform(options)

    similarities = cosine_similarity(
        prompt_vector,
        option_vectors
    )[0]

    order = np.argsort(similarities)[::-1]
    prediction = [option_labels[i] for i in order]

    tfidf_predictions.append(prediction)
    tfidf_scores.append(similarities)

In [19]:
train['tfidf_prediction'] = tfidf_predictions

tfidf_map = map_at_3(
    train['answer'].tolist(),
    tfidf_predictions
)

print('TF-IDF MAP@3:', round(tfidf_map, 4))

train[
    ['id', 'answer', 'tfidf_prediction']
].head(10)

TF-IDF MAP@3: 0.2762


,id,answer,tfidf_prediction
0,1,B,"[C, D, B, A, E]"
1,2,A,"[C, A, B, D, E]"
2,3,C,"[E, D, C, B, A]"
3,4,B,"[C, D, B, A, E]"
4,5,A,"[E, D, B, C, A]"
5,6,C,"[E, D, C, B, A]"
6,7,E,"[A, C, D, E, B]"
7,8,A,"[D, C, B, E, A]"
8,9,A,"[E, D, C, B, A]"
9,10,A,"[A, C, E, B, D]"


In [20]:
sentences = []

for column in clean_columns:
    for text in train[column]:
        sentences.append(text.split())

print('Number of sentences:', len(sentences))

Number of sentences: 12000


In [21]:
word2vec = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    seed=42
)

print('Word2Vec vocabulary size:', len(word2vec.wv))

Word2Vec vocabulary size: 3003


In [22]:
def get_sentence_vector(text):
    vectors = []

    for word in text.split():
        if word in word2vec.wv:
            vectors.append(word2vec.wv[word])

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

In [23]:
word2vec_predictions = []
word2vec_scores = []

for _, row in train.iterrows():
    prompt_vector = get_sentence_vector(
        row['prompt_clean']
    ).reshape(1, -1)

    option_vectors = []

    for label in option_labels:
        option_vector = get_sentence_vector(
            row[label + '_clean']
        )
        option_vectors.append(option_vector)

    option_vectors = np.array(option_vectors)

    similarities = cosine_similarity(
        prompt_vector,
        option_vectors
    )[0]

    order = np.argsort(similarities)[::-1]
    prediction = [option_labels[i] for i in order]

    word2vec_predictions.append(prediction)
    word2vec_scores.append(similarities)

In [24]:
train['word2vec_prediction'] = word2vec_predictions

word2vec_map = map_at_3(
    train['answer'].tolist(),
    word2vec_predictions
)

print('Word2Vec MAP@3:', round(word2vec_map, 4))

Word2Vec MAP@3: 0.3318


In [25]:
results = pd.DataFrame({
    'model': ['TF-IDF', 'Word2Vec'],
    'map@3': [tfidf_map, word2vec_map]
})

results

,model,map@3
0,TF-IDF,0.276167
1,Word2Vec,0.331750
